In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Initialize the Spark Session
# "local[*]" tells Spark to use all available CPU cores on your machine
spark = SparkSession.builder \
    .appName("GeoPulse_Local_Test") \
    .master("local[*]") \
    .getOrCreate()

# 2. Create some raw mock data (DeviceID, Lat, Long, Timestamp)
mock_data = [
    ("Device_123", 37.7749, -122.4194, "08:00"),
    ("Device_456", 37.7749, -122.4194, "08:05"),
    ("Device_123", 37.7750, -122.4190, "08:10"),
    ("Device_789", 34.0522, -118.2437, "08:15"),
    ("Device_123", 37.7755, -122.4180, "08:20")
]
columns = ["DeviceID", "Lat", "Long", "Time"]

# 3. Load the data into a distributed DataFrame
df = spark.read.csv(
    "Data/bengaluru_test_1m.csv",
    header=True,       # Uses the first row as column names
    inferSchema=True   # Automatically detects integers, doubles, and timestamps
)

print("\n--- Raw GPS Pings ---")
df.show()

# 4. Perform a transformation: Count how many times each device pinged
ping_counts = df.groupBy("DeviceID") \
    .agg(F.count("*").alias("Total_Pings")) \
    .orderBy(F.col("Total_Pings").desc())

print("\n--- Aggregated Ping Counts ---")
# 5. Action: spark doesn't actually compute anything until you call an action like show()
ping_counts.show()

# 6. Shut down the cluster
spark.stop()

In [6]:
import json
from pyspark.sql import functions as F
from sedona.spark import SedonaContext

def analyze_store_traffic_extended(
    csv_path,
    shop_lat,
    shop_lon,
    catchment_radius=500,
    visit_radius=20,
    min_pings=3,
):
    config = (
        SedonaContext.builder()
        .appName("GeoPulse_Sedona_Extended_Metrics")
        .config(
            "spark.jars.packages",
            "org.apache.sedona:sedona-spark-4.0_2.13:1.9.1,"
            "org.datasyslab:geotools-wrapper:1.9.1-33.5",
        )
        .config(
            "spark.jars.repositories",
            "https://artifacts.unidata.ucar.edu/repository/unidata-all",
        )
        .getOrCreate()
    )

    sedona = SedonaContext.create(config)

    try:
        print("Loading distributed data...")
        raw_df = sedona.read.csv(csv_path, header=True, inferSchema=True)

        # 1. Spatial points & distance calculation (cached so we only compute once)
        dist_df = (
            raw_df.withColumn(
                "device_point",
                F.expr("ST_Point(CAST(Longitude AS DOUBLE), CAST(Latitude AS DOUBLE))"),
            )
            .withColumn(
                "shop_point",
                F.expr(f"ST_Point(CAST({shop_lon} AS DOUBLE), CAST({shop_lat} AS DOUBLE))"),
            )
            .withColumn("Timestamp", F.to_timestamp(F.col("Timestamp")))
            .withColumn("distance_m", F.expr("ST_DistanceSphere(device_point, shop_point)"))
            .cache()
        )

        # 2. Broad Catchment metrics
        catchment_df = dist_df.filter(F.col("distance_m") <= catchment_radius)
        unique_catchment_devices = catchment_df.select("DeviceID").distinct().count()

        # 3. Storefront sessions (both pass-bys and confirmed visits)
        visit_df = dist_df.filter(F.col("distance_m") <= visit_radius)

        session_df = (
            visit_df.groupBy("DeviceID", F.window("Timestamp", "1 hour"))
            .agg(
                F.count("*").alias("ping_count"),
                F.min("Timestamp").alias("first_ping"),
                F.max("Timestamp").alias("last_ping"),
            )
            .withColumn(
                "dwell_time_min",
                (F.unix_timestamp("last_ping") - F.unix_timestamp("first_ping")) / 60.0,
            )
            .cache()
        )

        total_storefront_sessions = session_df.count()
        qualified_visits_df = session_df.filter(F.col("ping_count") >= min_pings).cache()

        # 4. Core Visit & Dwell Time Aggregations (single Spark pass)
        visit_stats = qualified_visits_df.agg(
            F.count("*").alias("total_visits"),
            F.countDistinct("DeviceID").alias("unique_customers"),
            F.avg("ping_count").alias("avg_pings"),
            F.avg("dwell_time_min").alias("avg_dwell_min"),
            F.expr("percentile_approx(dwell_time_min, 0.5)").alias("median_dwell_min"),
        ).first()

        total_visits = visit_stats["total_visits"] or 0
        unique_customers = visit_stats["unique_customers"] or 0
        passby_sessions = total_storefront_sessions - total_visits

        catchment_capture_pct = (
            round((unique_customers / unique_catchment_devices) * 100, 2)
            if unique_catchment_devices > 0
            else 0.0
        )
        storefront_conversion_pct = (
            round((total_visits / total_storefront_sessions) * 100, 2)
            if total_storefront_sessions > 0
            else 0.0
        )

        # 5. Customer Loyalty / Repeat Behavior
        customer_freq_df = qualified_visits_df.groupBy("DeviceID").agg(
            F.count("*").alias("visit_count")
        )
        loyalty_stats = customer_freq_df.agg(
            F.sum(F.when(F.col("visit_count") > 1, 1).otherwise(0)).alias("repeat_customers"),
            F.avg("visit_count").alias("avg_visits_per_customer"),
            F.max("visit_count").alias("max_visits_by_device"),
        ).first()

        repeat_customers = loyalty_stats["repeat_customers"] or 0
        repeat_customer_pct = (
            round((repeat_customers / unique_customers) * 100, 2)
            if unique_customers > 0
            else 0.0
        )

        # 6. Temporal Patterns: Hourly distribution & Peak Hour
        hourly_rows = (
            qualified_visits_df.withColumn("hour", F.hour(F.col("window.start")))
            .groupBy("hour")
            .agg(F.count("*").alias("visits"))
            .orderBy("hour")
            .collect()
        )
        hourly_distribution = {f"{row['hour']:02d}:00": row["visits"] for row in hourly_rows}
        peak_hour = (
            f"{max(hourly_rows, key=lambda r: r['visits'])['hour']:02d}:00"
            if hourly_rows
            else None
        )

        # 7. Weekday vs. Weekend split (In Spark SQL, 1 = Sunday, 7 = Saturday)
        day_type_rows = (
            qualified_visits_df.withColumn(
                "day_type",
                F.when(F.dayofweek(F.col("window.start")).isin([1, 7]), "weekend").otherwise("weekday"),
            )
            .groupBy("day_type")
            .agg(F.count("*").alias("visits"))
            .collect()
        )
        day_type_map = {row["day_type"]: row["visits"] for row in day_type_rows}
        weekday_visits = day_type_map.get("weekday", 0)
        weekend_visits = day_type_map.get("weekend", 0)

        # 8. Observed Trade Area Pull
        confirmed_ids = qualified_visits_df.select("DeviceID").distinct()
        pull_row = (
            dist_df.join(confirmed_ids, on="DeviceID", how="inner")
            .groupBy("DeviceID")
            .agg(F.max("distance_m").alias("max_observed_dist_m"))
            .agg(
                F.expr("percentile_approx(max_observed_dist_m, 0.5)").alias("median_pull_m"),
                F.expr("percentile_approx(max_observed_dist_m, 0.75)").alias("p75_pull_m"),
            )
            .first()
        )

        return {
            "config": {
                "shop_lat": shop_lat,
                "shop_lon": shop_lon,
                "catchment_radius_m": catchment_radius,
                "visit_radius_m": visit_radius,
                "min_pings": min_pings,
            },
            "catchment_and_conversion": {
                "unique_devices_in_catchment": unique_catchment_devices,
                "storefront_passby_sessions": passby_sessions,
                "catchment_capture_rate_pct": catchment_capture_pct,
                "storefront_conversion_rate_pct": storefront_conversion_pct,
            },
            "visits_and_engagement": {
                "actual_unique_customers": unique_customers,
                "total_visits_including_repeats": total_visits,
                "average_pings_per_visit": round(visit_stats["avg_pings"] or 0, 1),
                "average_dwell_time_min": round(visit_stats["avg_dwell_min"] or 0, 1),
                "median_dwell_time_min": round(visit_stats["median_dwell_min"] or 0, 1),
            },
            "loyalty": {
                "repeat_customers": repeat_customers,
                "repeat_customer_rate_pct": repeat_customer_pct,
                "avg_visits_per_customer": round(loyalty_stats["avg_visits_per_customer"] or 0, 2),
                "max_visits_by_single_device": loyalty_stats["max_visits_by_device"] or 0,
            },
            "temporal_patterns": {
                "peak_visit_hour": peak_hour,
                "weekday_visits": weekday_visits,
                "weekend_visits": weekend_visits,
                "weekend_share_pct": round((weekend_visits / total_visits) * 100, 2) if total_visits > 0 else 0.0,
                "hourly_visit_distribution": hourly_distribution,
            },
            "trade_area_pull": {
                "median_customer_reach_m": round(pull_row["median_pull_m"] or 0, 1) if pull_row else 0.0,
                "p75_customer_reach_m": round(pull_row["p75_pull_m"] or 0, 1) if pull_row else 0.0,
            },
        }
    finally:
        sedona.stop()

metrics = analyze_store_traffic_extended(
    "/content/drive/MyDrive/Colab Notebooks/Data/GeoPulse_Infotact_Internship/bengaluru_test_1m.csv",
    12.978973,
    77.64527,
)
print(json.dumps(metrics, indent=2))

Loading distributed data...
{
  "config": {
    "shop_lat": 12.978973,
    "shop_lon": 77.64527,
    "catchment_radius_m": 500,
    "visit_radius_m": 20,
    "min_pings": 3
  },
  "catchment_and_conversion": {
    "unique_devices_in_catchment": 441,
    "storefront_passby_sessions": 275,
    "catchment_capture_rate_pct": 4.54,
    "storefront_conversion_rate_pct": 7.41
  },
  "visits_and_engagement": {
    "actual_unique_customers": 20,
    "total_visits_including_repeats": 22,
    "average_pings_per_visit": 3.0,
    "average_dwell_time_min": 35.1,
    "median_dwell_time_min": 32.1
  },
  "loyalty": {
    "repeat_customers": 2,
    "repeat_customer_rate_pct": 10.0,
    "avg_visits_per_customer": 1.1,
    "max_visits_by_single_device": 2
  },
  "temporal_patterns": {
    "peak_visit_hour": "07:00",
    "weekday_visits": 8,
    "weekend_visits": 14,
    "weekend_share_pct": 63.64,
    "hourly_visit_distribution": {
      "04:00": 1,
      "05:00": 2,
      "06:00": 2,
      "07:00": 4,
 

### Cannibalization Metrics

In [8]:
import json
import math
import pyspark
from pyspark.sql import functions as F
from sedona.spark import SedonaContext


def _haversine_m(lat1, lon1, lat2, lon2):
    """Exact straight-line spherical distance in meters between the two store pins."""
    r = 6_371_008.8
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return round(2 * r * math.atan2(math.sqrt(a), math.sqrt(1 - a)), 1)


def _get_confirmed_visits(dist_df, dist_col, visit_radius, min_pings):
    """Extracts qualified hourly store visits and per-device visit counts."""
    sessions = (
        dist_df.filter(F.col(dist_col) <= visit_radius)
        .groupBy("DeviceID", F.window("Timestamp", "1 hour"))
        .agg(F.count("*").alias("ping_count"))
        .filter(F.col("ping_count") >= min_pings)
    )
    return sessions.groupBy("DeviceID").agg(F.count("*").alias("visits"))


def analyze_two_store_cannibalization(
    csv_path,
    store_a,
    store_b,
    mode="expansion",               # "expansion" (A existing, B proposed) or "existing_pair" (both open)
    catchment_radius=500,
    visit_radius=20,
    min_pings=3,
    huff_beta=2.0,                  # Distance decay exponent (1.5-2.0 for daily/urban retail)
    attractiveness_a=1.0,           # Relative store size/appeal weight for Store A
    attractiveness_b=1.0,           # Relative store size/appeal weight for Store B
    avg_spend_per_visit=250.0,      # Optional financial inputs (e.g., INR per visit)
    profit_margin_pct=0.30,         # Contribution margin (30% = 0.30)
    new_store_fixed_cost=0.0,       # Period fixed overhead for Store B (lease + staff)
):
    spark_mm = ".".join(pyspark.__version__.split(".")[:2])
    scala_ver = "2.13" if int(spark_mm.split(".")[0]) >= 4 else "2.12"

    config = (
        SedonaContext.builder()
        .appName("GeoPulse_Cannibalization_Analysis")
        .config(
            "spark.jars.packages",
            f"org.apache.sedona:sedona-spark-{spark_mm}_{scala_ver}:1.9.1,"
            "org.datasyslab:geotools-wrapper:1.9.1-33.5",
        )
        .config(
            "spark.jars.repositories",
            "https://artifacts.unidata.ucar.edu/repository/unidata-all",
        )
        .getOrCreate()
    )

    sedona = SedonaContext.create(config)
    dist_df = None

    try:
        print(f"Loading data for two-store cannibalization ({mode})...")
        raw_df = sedona.read.csv(csv_path, header=True, inferSchema=True)

        # 1. Single-pass spatial distance calculation to both stores
        dist_df = (
            raw_df.withColumn("Latitude", F.col("Latitude").cast("double"))
            .withColumn("Longitude", F.col("Longitude").cast("double"))
            .withColumn("Timestamp", F.to_timestamp(F.col("Timestamp")))
            .filter(
                F.col("DeviceID").isNotNull()
                & F.col("Latitude").isNotNull()
                & F.col("Longitude").isNotNull()
                & F.col("Timestamp").isNotNull()
            )
            .withColumn("device_point", F.expr("ST_Point(Longitude, Latitude)"))
            .withColumn("store_a_pt", F.expr(f"ST_Point({float(store_a['lon'])}, {float(store_a['lat'])})"))
            .withColumn("store_b_pt", F.expr(f"ST_Point({float(store_b['lon'])}, {float(store_b['lat'])})"))
            .withColumn("dist_a_m", F.expr("ST_DistanceSphere(device_point, store_a_pt)"))
            .withColumn("dist_b_m", F.expr("ST_DistanceSphere(device_point, store_b_pt)"))
            .cache()
        )

        store_separation_m = _haversine_m(
            store_a["lat"], store_a["lon"], store_b["lat"], store_b["lon"]
        )

        # 2. Catchment sets (Umbrex Proximity Overlap + Esri Directional Overlap)
        catchment_a = dist_df.filter(F.col("dist_a_m") <= catchment_radius).select("DeviceID").distinct()
        catchment_b = dist_df.filter(F.col("dist_b_m") <= catchment_radius).select("DeviceID").distinct()

        catchment_a_cnt = catchment_a.count()
        catchment_b_cnt = catchment_b.count()
        shared_catchment_cnt = catchment_a.intersect(catchment_b).count()
        union_catchment_cnt = catchment_a_cnt + catchment_b_cnt - shared_catchment_cnt

        proximity_overlap_pct = (
            round((shared_catchment_cnt / union_catchment_cnt) * 100, 2)
            if union_catchment_cnt > 0 else 0.0
        )
        esri_a_within_b_pct = (
            round((shared_catchment_cnt / catchment_a_cnt) * 100, 2)
            if catchment_a_cnt > 0 else 0.0
        )
        esri_b_within_a_pct = (
            round((shared_catchment_cnt / catchment_b_cnt) * 100, 2)
            if catchment_b_cnt > 0 else 0.0
        )

        # 3. Store A confirmed customers & baseline visits
        visits_a_df = _get_confirmed_visits(dist_df, "dist_a_m", visit_radius, min_pings).cache()
        stats_a = visits_a_df.agg(
            F.count("*").alias("unique_customers"),
            F.sum("visits").alias("total_visits"),
        ).first()
        customers_a_cnt = stats_a["unique_customers"] or 0
        visits_a_total = stats_a["total_visits"] or 0

        # Store A's observed catchment capture rate (visits per catchment device)
        capture_rate_a = (customers_a_cnt / catchment_a_cnt) if catchment_a_cnt > 0 else 0.0
        visits_per_customer_a = (visits_a_total / customers_a_cnt) if customers_a_cnt > 0 else 0.0

        # 4. Device activity centroids (proxy for where each customer usually operates)
        device_centroids = (
            dist_df.groupBy("DeviceID")
            .agg(
                F.avg("Latitude").alias("avg_lat"),
                F.avg("Longitude").alias("avg_lon"),
            )
            .withColumn("centroid_pt", F.expr("ST_Point(avg_lon, avg_lat)"))
            .withColumn(
                "centroid_dist_a_m",
                F.greatest(
                    F.expr(f"ST_DistanceSphere(centroid_pt, ST_Point({float(store_a['lon'])}, {float(store_a['lat'])}))"),
                    F.lit(10.0),  # floor at 10m to prevent division-by-zero
                ),
            )
            .withColumn(
                "centroid_dist_b_m",
                F.greatest(
                    F.expr(f"ST_DistanceSphere(centroid_pt, ST_Point({float(store_b['lon'])}, {float(store_b['lat'])}))"),
                    F.lit(10.0),
                ),
            )
        )

        # 5. Huff Gravity defection model on Store A's customers exposed to Store B's catchment
        exposed_a_df = (
            visits_a_df.join(catchment_b, on="DeviceID", how="inner")
            .join(device_centroids, on="DeviceID", how="inner")
            .withColumn("utility_a", F.lit(float(attractiveness_a)) * F.pow(F.col("centroid_dist_a_m"), -float(huff_beta)))
            .withColumn("utility_b", F.lit(float(attractiveness_b)) * F.pow(F.col("centroid_dist_b_m"), -float(huff_beta)))
            .withColumn("p_switch_to_b", F.col("utility_b") / (F.col("utility_a") + F.col("utility_b")))
            .withColumn("expected_transferred_visits", F.col("visits") * F.col("p_switch_to_b"))
        )

        huff_stats = exposed_a_df.agg(
            F.count("*").alias("exposed_customers"),
            F.sum("visits").alias("exposed_visits"),
            F.avg("p_switch_to_b").alias("avg_defection_prob"),
            F.sum("p_switch_to_b").alias("expected_defected_customers"),
            F.sum("expected_transferred_visits").alias("expected_transferred_visits"),
        ).first()

        exposed_customers_a = huff_stats["exposed_customers"] or 0
        expected_transferred_visits = round(huff_stats["expected_transferred_visits"] or 0.0, 1)
        expected_defected_customers = round(huff_stats["expected_defected_customers"] or 0.0, 1)
        avg_defection_prob_pct = round((huff_stats["avg_defection_prob"] or 0.0) * 100, 2)

        # 6. Branch metrics for Expansion vs. Existing Pair
        cross_visitation_block = None
        if mode == "existing_pair":
            visits_b_df = _get_confirmed_visits(dist_df, "dist_b_m", visit_radius, min_pings).cache()
            stats_b = visits_b_df.agg(
                F.count("*").alias("unique_customers"),
                F.sum("visits").alias("total_visits"),
            ).first()
            customers_b_cnt = stats_b["unique_customers"] or 0
            visits_b_total = stats_b["total_visits"] or 0

            shared_visitors_cnt = (
                visits_a_df.select("DeviceID").intersect(visits_b_df.select("DeviceID")).count()
            )
            union_visitors_cnt = customers_a_cnt + customers_b_cnt - shared_visitors_cnt

            cross_visitation_block = {
                "store_a_confirmed_customers": customers_a_cnt,
                "store_b_confirmed_customers": customers_b_cnt,
                "shared_confirmed_customers": shared_visitors_cnt,
                "placer_cross_visit_a_to_b_pct": round((shared_visitors_cnt / customers_a_cnt) * 100, 2) if customers_a_cnt else 0.0,
                "placer_cross_visit_b_to_a_pct": round((shared_visitors_cnt / customers_b_cnt) * 100, 2) if customers_b_cnt else 0.0,
                "confirmed_customer_jaccard_overlap_pct": round((shared_visitors_cnt / union_visitors_cnt) * 100, 2) if union_visitors_cnt else 0.0,
            }
            projected_visits_b = float(visits_b_total)
            visits_b_df.unpersist()
        else:
            # Estimate Store B visits by applying Store A's proven capture & frequency to B's net-new catchment
            # plus the visits expected to transfer from Store A via Huff Gravity
            net_new_catchment_b = max(0, catchment_b_cnt - exposed_customers_a)
            projected_net_new_visits_b = round(net_new_catchment_b * capture_rate_a * visits_per_customer_a, 1)
            projected_visits_b = round(projected_net_new_visits_b + expected_transferred_visits, 1)

        # 7. Dual Cannibalization Rates (Umbrex vs. GrowthFactor/PassBy) + Net Incremental
        net_incremental_visits = round(max(0.0, projected_visits_b - expected_transferred_visits), 1)

        umbrex_erosion_rate_pct = (
            round((expected_transferred_visits / visits_a_total) * 100, 2)
            if visits_a_total > 0 else 0.0
        )
        growthfactor_cannibalization_rate_pct = (
            round((expected_transferred_visits / projected_visits_b) * 100, 2)
            if projected_visits_b > 0 else 0.0
        )

        # Risk verdict using Umbrex (20% erosion) & PassBy (40% overlap) benchmarks
        if umbrex_erosion_rate_pct > 20.0 or esri_b_within_a_pct > 40.0:
            risk_level = "HIGH_RISK"
        elif umbrex_erosion_rate_pct >= 10.0:
            risk_level = "MODERATE_RISK_CAPACITY_RELIEF_ZONE"
        else:
            risk_level = "LOW_RISK_STRONG_EXPANSION"

        projected_new_store_sales = round(projected_visits_b * avg_spend_per_visit, 2)
        cannibalized_sales_from_a = round(expected_transferred_visits * avg_spend_per_visit, 2)
        incremental_sales = round(projected_new_store_sales - cannibalized_sales_from_a, 2)
        net_profit_impact = round((incremental_sales * profit_margin_pct) - new_store_fixed_cost, 2)

        visits_a_df.unpersist()

        result = {
            "stores": {
                "mode": mode,
                "store_a": store_a,
                "store_b": store_b,
                "store_separation_m": store_separation_m,
                "catchment_radius_m": catchment_radius,
            },
            "trade_area_overlap": {
                "store_a_catchment_devices": catchment_a_cnt,
                "store_b_catchment_devices": catchment_b_cnt,
                "shared_catchment_devices": shared_catchment_cnt,
                "net_new_catchment_devices_for_b": catchment_b_cnt - shared_catchment_cnt,
                "umbrex_proximity_overlap_pct": proximity_overlap_pct,
                "esri_store_a_within_b_pct": esri_a_within_b_pct,
                "esri_store_b_within_a_pct": esri_b_within_a_pct,
            },
            "huff_gravity_defection": {
                "store_a_baseline_customers": customers_a_cnt,
                "store_a_baseline_visits": visits_a_total,
                "store_a_customers_exposed_to_b": exposed_customers_a,
                "avg_defection_probability_pct": avg_defection_prob_pct,
                "expected_defected_customers": expected_defected_customers,
                "expected_transferred_visits": expected_transferred_visits,
            },
            "cannibalization_metrics": {
                "projected_or_actual_store_b_visits": projected_visits_b,
                "net_incremental_visits": net_incremental_visits,
                "umbrex_existing_store_erosion_pct": umbrex_erosion_rate_pct,
                "growthfactor_new_store_cannibalization_pct": growthfactor_cannibalization_rate_pct,
                "risk_verdict": risk_level,
            },
            "financial_impact_estimate": {
                "assumptions": {
                    "avg_spend_per_visit": avg_spend_per_visit,
                    "profit_margin_pct": profit_margin_pct,
                    "new_store_fixed_cost": new_store_fixed_cost,
                },
                "projected_store_b_sales": projected_new_store_sales,
                "cannibalized_sales_from_a": cannibalized_sales_from_a,
                "umbrex_incremental_sales": incremental_sales,
                "net_profit_impact": net_profit_impact,
            },
        }

        if cross_visitation_block:
            result["existing_pair_cross_visitation"] = cross_visitation_block

        return result
    finally:
        if dist_df is not None:
            dist_df.unpersist()
        sedona.stop()


# Example: Existing Indiranagar Store A vs. Proposed Site B ~650m away
report = analyze_two_store_cannibalization(
    csv_path="/content/drive/MyDrive/Colab Notebooks/Data/GeoPulse_Infotact_Internship/bengaluru_test_1m.csv",
    store_a={"name": "Store A (Existing)", "lat": 12.978973, "lon": 77.64527},
    store_b={"name": "Store B (Candidate)", "lat": 12.973500, "lon": 77.64120},
    mode="expansion",
    catchment_radius=500,
    visit_radius=20,
    min_pings=3,
    avg_spend_per_visit=300.0,
    profit_margin_pct=0.35,
    new_store_fixed_cost=150000.0,
)
print(json.dumps(report, indent=2))

Loading data for two-store cannibalization (expansion)...
{
  "stores": {
    "mode": "expansion",
    "store_a": {
      "name": "Store A (Existing)",
      "lat": 12.978973,
      "lon": 77.64527
    },
    "store_b": {
      "name": "Store B (Candidate)",
      "lat": 12.9735,
      "lon": 77.6412
    },
    "store_separation_m": 751.6,
    "catchment_radius_m": 500
  },
  "trade_area_overlap": {
    "store_a_catchment_devices": 441,
    "store_b_catchment_devices": 317,
    "shared_catchment_devices": 254,
    "net_new_catchment_devices_for_b": 63,
    "umbrex_proximity_overlap_pct": 50.4,
    "esri_store_a_within_b_pct": 57.6,
    "esri_store_b_within_a_pct": 80.13
  },
  "huff_gravity_defection": {
    "store_a_baseline_customers": 20,
    "store_a_baseline_visits": 22,
    "store_a_customers_exposed_to_b": 10,
    "avg_defection_probability_pct": 50.47,
    "expected_defected_customers": 5.0,
    "expected_transferred_visits": 6.2
  },
  "cannibalization_metrics": {
    "project

### Exporting as json file

In [ ]:
import json
from google.colab import files

local_path = "cannibalization_report.json"
with open(local_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, default=str)

files.download(local_path)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
!pip install -q apache-sedona==1.9.1